# Segmento 2: Function Calling

Diamo all'LLM un **tool** per cercare ricette. Sarà lui a decidere quando usarlo.

Il contesto della conversazione viene mantenuto. Vediamo se "dammene un'altra" ora funziona.

In [15]:
from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient
import json, os

load_dotenv()
client = OpenAI()
qdrant = QdrantClient(host="localhost", port=6333)

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-5.4-nano"
COLLECTION = "recipes"

# Verifichiamo che Qdrant abbia i dati della sessione 2
if not qdrant.collection_exists(COLLECTION):
    print(f"La collection '{COLLECTION}' non esiste!")
    print("Vai nel notebook session2/segment_2.ipynb ed eseguilo per crearla.")
else:
    info = qdrant.get_collection(COLLECTION)
    print(f"Collection '{COLLECTION}': {info.points_count} ricette")
    print(f"Modello chat: {CHAT_MODEL}")

Collection 'recipes': 1000 ricette
Modello chat: gpt-5.4-nano


## Definiamo il tool per l'LLM

Lo schema dice all'LLM: *"hai a disposizione una funzione `retrieve_recipes` che accetta una query e restituisce ricette"*.

L'LLM non esegue la funzione, ci dice **cosa vuole chiamare** e con quali argomenti.

In [16]:
# Lo schema del tool — è quello che l'LLM "vede"
tools = [
    {
        "type": "function",
        "function": {
            "name": "retrieve_recipes",
            "description": "Cerca ricette nel database per similarità semantica. Usa questo tool quando l'utente chiede ricette o suggerimenti culinari.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "La query di ricerca per trovare ricette rilevanti"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

# La funzione vera, quella che eseguiamo noi
def retrieve_recipes(query, top_n=5):
    """Cerca le ricette più rilevanti in Qdrant."""
    response = client.embeddings.create(input=[query], model=EMBED_MODEL)
    query_vector = response.data[0].embedding

    results = qdrant.query_points(
        collection_name=COLLECTION,
        query=query_vector,
        limit=top_n,
        with_payload=True,
    )

    recipes = []
    for point in results.points:
        recipes.append({
            "title": point.payload["title"],
            "ingredients": point.payload["ingredients"],
            "instructions": point.payload["instructions"],
            "score": round(point.score, 4),
        })
    return recipes

print("Tool definito: retrieve_recipes")
print(f"Schema:\n{json.dumps(tools[0]['function'], indent=2)}")

Tool definito: retrieve_recipes
Schema:
{
  "name": "retrieve_recipes",
  "description": "Cerca ricette nel database per similarit\u00e0 semantica. Usa questo tool quando l'utente chiede ricette o suggerimenti culinari.",
  "parameters": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string",
        "description": "La query di ricerca per trovare ricette rilevanti"
      }
    },
    "required": [
      "query"
    ]
  }
}


In [23]:
# System prompt per l'agente
SYSTEM_PROMPT = """Sei un assistente culinario.
Aiuti gli utenti a trovare ricette dal nostro database.
Usa il tool retrieve_recipes quando l'utente cerca ricette o suggerimenti culinari.
"""

# Inizializziamo la conversazione
messages = [
    {"role": "system", "content": SYSTEM_PROMPT}
]
print("Conversazione inizializzata")

Conversazione inizializzata


## Prima domanda

Mandiamo la stessa query del notebook 1, ma questa volta l'LLM ha un tool a disposizione.

In [24]:
# Turno 1: l'utente chiede una ricetta
messages.append(
    {
        "role": "user",
        "content": "Cerco una ricetta con pomodoro"
    }
)

response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=messages,
    tools=tools,
)

assistant_msg = response.choices[0].message

# L'LLM non ha risposto con testo: ha chiesto di chiamare un tool!
print("=== Tool Call — JSON raw ===\n")
for tc in assistant_msg.tool_calls:
    print(json.dumps({
        "id": tc.id,
        "function": {
            "name": tc.function.name,
            "arguments": tc.function.arguments
        }
    }, indent=2))

print("\n=== Versione leggibile ===\n")
for tc in assistant_msg.tool_calls:
    args = json.loads(tc.function.arguments)
    print(f"L'LLM vuole chiamare: {tc.function.name}()")
    for k, v in args.items():
        print(f"  {k}: {v}")

=== Tool Call — JSON raw ===

{
  "id": "call_joVoZIx4p81ShKB2MtjTGmqp",
  "function": {
    "name": "retrieve_recipes",
    "arguments": "{\"query\":\"ricetta con pomodoro\"}"
  }
}

=== Versione leggibile ===

L'LLM vuole chiamare: retrieve_recipes()
  query: ricetta con pomodoro


## Eseguiamo il tool e restituiamo il risultato

L'LLM ci ha detto **cosa** vuole fare. Ora tocca a noi **eseguire** la funzione e mandargli indietro i risultati.

In [25]:
# Eseguiamo il tool call
tool_call = assistant_msg.tool_calls[0]
args = json.loads(tool_call.function.arguments)
result = retrieve_recipes(**args)

print("Risultati della ricerca:\n")
for r in result:
    print(f"  {r['score']:.4f}  {r['title']}")

# Aggiungiamo alla conversazione: il messaggio dell'assistente + il risultato del tool
messages.append(assistant_msg)
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(result)
})

# Ora l'LLM genera la risposta finale con i dati reali
response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=messages,
    tools=tools,
)

final_msg = response.choices[0].message
messages.append(final_msg)

print(f"\n--- Risposta dell'assistente ---\n\n{final_msg.content}")

Risultati della ricerca:

  0.6228  Pappa al Pomodoro
  0.5379  Classic Tomato Sauce
  0.5273  Tomato Soup with Arugula, Croutons, and Pecorino
  0.5107  Mozzarella Arrabiata Salsa
  0.5007  Anchovies in Tomato Sauce with Pasta

--- Risposta dell'assistente ---

Certo! Nel nostro database ci sono alcune ricette con il **pomodoro**. Quale ti ispira di più?

1) **Pappa al Pomodoro**
- **Ingredienti (sintesi):** pomodori, aglio, semi di finocchio, olio d’oliva, basilico, pane
- **Idea:** zuppa/crema rustica e molto saporita.

2) **Classic Tomato Sauce**
- **Ingredienti (sintesi):** olio d’oliva, cipolla, alloro, origano, aglio, passata/conserva pomodoro, concentrato
- **Idea:** sugo classico da pasta (ottimo anche per lasagne).

3) **Tomato Soup with Arugula, Croutons, and Pecorino**
- **Ingredienti (sintesi):** pomodori, cipolla, timo, rucola, crostini, pecorino
- **Idea:** vellutata più “fresca” e con guarnizione.

4) **Mozzarella Arrabiata Salsa**
- **Ingredienti (sintesi):** pomodori,

## Il momento della verità: "dammene un'altra"

Stessa richiesta che nel notebook 1 dava risultati senza senso.

Questa volta l'LLM ha il **contesto della conversazione**, sa che stavamo parlando di ricette con pomodoro.

In [32]:
# Turno 2: la richiesta che prima falliva
messages.append({"role": "user", "content": "Grazie"})

response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=messages,
    tools=tools,
)

assistant_msg_2 = response.choices[0].message

# Vediamo cosa ha fatto l'LLM
if assistant_msg_2.tool_calls:
    print("L'LLM ha fatto un nuovo tool call!\n")
    for tc in assistant_msg_2.tool_calls:
        args = json.loads(tc.function.arguments)
        print(f"  Funzione: {tc.function.name}")
        print(f"  Argomenti: {args}")
        print(f"\n  Ha capito dal contesto che 'un'altra' si riferisce a ricette con pomodoro!")
else:
    print("L'LLM ha risposto direttamente (aveva già risultati dal turno precedente):\n")
    print(assistant_msg_2.content)

L'LLM ha risposto direttamente (aveva già risultati dal turno precedente):

Prego! 😊 Se vuoi dimmi anche cosa hai in casa (ingredienti o attrezzatura) e ti aiuto a scegliere la ricetta perfetta e a organizzarla passo‑passo.


In [ ]:
# Completiamo il turno 2
if assistant_msg_2.tool_calls:
    tool_call_2 = assistant_msg_2.tool_calls[0]
    args_2 = json.loads(tool_call_2.function.arguments)
    result_2 = retrieve_recipes(**args_2)

    print("Risultati:\n")
    for r in result_2:
        print(f"  {r['score']:.4f}  {r['title']}")

    messages.append(assistant_msg_2)
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call_2.id,
        "content": json.dumps(result_2)
    })

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        tools=tools,
    )
    final_msg_2 = response.choices[0].message
    messages.append(final_msg_2)
    print(f"\n--- Risposta dell'assistente ---\n\n{final_msg_2.content}")
else:
    messages.append(assistant_msg_2)
    print("(Risposta già mostrata sopra)")

(Risposta già mostrata sopra)


## Cosa abbiamo visto

Con il function calling:

1. L'LLM **decide da solo** quando cercare nel database
2. Il **contesto della conversazione** viene mantenuto, "dammene un'altra" funziona
3. L'LLM **non esegue** codice, ci dice cosa vuole fare e noi eseguiamo

Prossimo passo: aggiungiamo un secondo tool per fare qualcosa **nel mondo reale**, torniamo alle slides.